# 02 — Linear Regression: Interpretation

**Covers concept IDs:** B3, B4a–c, B5, B6, B7, B8.

## Learning outcomes
1. Read a regression output in any of the four log/level combinations of $x$ and $y$.
2. Interpret a categorical coefficient against its *reference level*.
3. Interpret an interaction coefficient as a **change of slope**.
4. Diagnose **omitted-variable bias** by sign.
5. Compute $R^2$, MSE, and RSS by hand and spot overfitting from train-vs-test gap.
6. Understand why a predictive coefficient is **not causal**.

## 1. Coefficient interpretation — the four cases

Let the fitted coefficient on the first variable be $\beta_1$.

| Model | Interpretation of $\beta_1$ |
|---|---|
| $y = \beta_0 + \beta_1 x + \ldots$ | 1 unit ↑ in $x$ ⇒ $\beta_1$ units change in $y$. |
| $\log y = \beta_0 + \beta_1 x + \ldots$ | 1 unit ↑ in $x$ ⇒ $\approx 100\,\beta_1\%$ change in $y$. |
| $y = \beta_0 + \beta_1 \log x + \ldots$ | 1% ↑ in $x$ ⇒ $\beta_1/100$ unit change in $y$. |
| $\log y = \beta_0 + \beta_1 \log x + \ldots$ | 1% ↑ in $x$ ⇒ $\beta_1\%$ change in $y$ — this is an **elasticity**. |

**Memory device.** *Log on the left* makes the *y-side* a percent; *log on the right* makes the *x-side* a percent.

## 2. Categorical features and the reference level

`pd.get_dummies(df, drop_first=True)` drops one level; the intercept absorbs it.

**Example.** Regress `price ~ brand` where `brand ∈ {minute.maid, tropicana, dominicks}` and `dominicks` is the reference.
$$\hat{\text{price}} = \underbrace{1.90}_{\beta_0\,=\,\text{Dominick's mean}} + \underbrace{0.35}_{\beta_{\text{MM}}}\mathbb 1_{\text{MM}} + \underbrace{0.95}_{\beta_{\text{Tro}}}\mathbb 1_{\text{Tro}}.$$

Interpret: Minute Maid is \$0.35 **more expensive than Dominick's**; Tropicana is \$0.95 **more expensive than Dominick's**. The reference level is the baseline — the intercept *is* the mean for that level (when there are no other covariates).

## 3. Interactions — a change of slope

Model `int_rate ~ loan_amnt * term` expands to
$$\text{int\_rate} = \beta_0 + \beta_1 \text{loan\_amnt} + \beta_2 \mathbb 1[\text{term}=60] + \beta_3\,\text{loan\_amnt}\cdot\mathbb 1[\text{term}=60].$$

For a **36-month** loan (reference), the slope on `loan_amnt` is $\beta_1$.

For a **60-month** loan, the slope is $\beta_1 + \beta_3$, and the intercept shifts by $\beta_2$.

In Assignment 2 Q3: $\beta_1=-0.00004$ (essentially flat), $\beta_3=+0.00103$, so for long loans an extra \$1,000 raises the rate by ≈ 0.1pp; for short loans it does nothing.

**Exam move.** An interaction question is really asking: *"what is the slope for each subgroup?"* Always write out both simple slopes.

## 3a. Polynomial / log transforms
$x^2$ or $\log x$ are just *new columns* of the design matrix — the model remains **linear in $\beta$**, which is all OLS needs.

## 4. Omitted-variable bias (OVB)

If the *true* model is $y = \beta_1 x_1 + \beta_2 x_2 + \varepsilon$ and we regress $y$ on $x_1$ alone, the estimated $\hat\beta_1$ is biased by
$$E[\hat\beta_1] - \beta_1 \;\approx\; \beta_2 \cdot \text{Cov}(x_1,x_2)/\text{Var}(x_1).$$

So the sign of the bias is
$$\text{sign}(\text{bias}) = \text{sign}(\beta_2)\times\text{sign}(\text{corr}(x_1,x_2)).$$

### Worked example from Assignment 2
Regress `int_rate ~ loan_amnt` ⇒ $\hat\beta = 0.00114$.
Regress `int_rate ~ loan_amnt + log_income` ⇒ $\hat\beta = 0.00148$.

Why does the coefficient **grow** when we add income?
- Income enters with a **negative** coefficient (richer borrowers pay less).
- Loan size is **positively correlated** with income (richer people take bigger loans).
- When we omit income, part of income's *negative* effect leaks into loan size, biasing the coefficient *downward* (toward 0).
- Adding income back lets `loan_amnt` reveal its true positive effect on rate.

Sign check: $\text{sign}(\beta_{\text{income}})\times\text{sign}(\text{corr}(\text{loan\_amnt},\text{income})) = (-)(+) = (-)$. The bias in the short regression was negative, so the omitted-variable model under-estimated the true coefficient. ✓

## 5. Confounding = OVB for causal claims

A predictive coefficient is **not** a causal effect unless every variable correlated with both the treatment $x^\star$ and the outcome $y$ has been controlled for. In the OJ lecture example, one could not treat the coefficient on `price` as *the* causal price sensitivity unless shelf-placement, promotional calendar, brand loyalty, etc., were all in the regression.

## 6. Goodness of fit

$$\text{RSS} = \sum_{i=1}^n (\hat y_i-y_i)^2,\qquad \text{TSS} = \sum_{i=1}^n(y_i-\bar y)^2,\qquad \text{MSE} = \tfrac{\text{RSS}}{n},\qquad R^2 = 1 - \tfrac{\text{RSS}}{\text{TSS}}.$$

- $R^2$ is the **fraction of variance** in $y$ explained by the fitted values.
- $R^2 \in [0,1]$ on training data; can be *negative* on test data if the model is worse than predicting the training-mean.
- **Overfitting signature:** training $R^2 \gg$ test $R^2$ (Midterm Practice MC 1).

### Numerical drill
Given RSS = 200 and TSS = 1000: $R^2 = 1 - 200/1000 = 0.80$. If $n=50$ then MSE = $200/50 = 4$.

## 7. Train/test split and prediction

- Fit on training set; compute in-sample RSS and $R^2$.
- Apply fitted $\hat\beta$ to test set to get $\hat y_{\text{new}} = X_{\text{new}}\hat\beta$ and a test RSS/$R^2$.
- Never touch the test set while tuning!
- **Extrapolation risk:** predictions for $x$ outside the training support are unreliable — the fit has no evidence there.

## 8. Small code demo (**not exam material**)

In [1]:
import numpy as np

# tiny dataset: y = 2 + 3x + noise
np.random.seed(0)
x = np.linspace(0, 10, 40)
y = 2 + 3*x + np.random.normal(0, 1.5, size=x.shape)

# fit OLS via the normal equations
X = np.column_stack([np.ones_like(x), x])
beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
y_hat = X @ beta_hat

RSS = ((y - y_hat)**2).sum()
TSS = ((y - y.mean())**2).sum()
R2 = 1 - RSS/TSS
MSE = RSS / len(y)
print(f"beta_hat = {beta_hat.round(3)}")
print(f"RSS = {RSS:.2f}   TSS = {TSS:.2f}   R^2 = {R2:.3f}   MSE = {MSE:.3f}")

# Deliberate OVB: true model has two drivers; short regression omits one.
x1 = np.random.normal(size=500)
x2 = 0.6*x1 + np.random.normal(scale=0.8, size=500)   # x2 correlated with x1
y  = 1 + 0.5*x1 - 1.0*x2 + np.random.normal(scale=0.5, size=500)   # beta_2 = -1

short = np.linalg.lstsq(np.column_stack([np.ones(500), x1]),      y, rcond=None)[0]
full  = np.linalg.lstsq(np.column_stack([np.ones(500), x1, x2]),  y, rcond=None)[0]
print(f"\nOVB demonstration: true beta_1 = 0.5")
print(f"Short regression (only x1)      beta_1 ≈ {short[1]:.3f}  (biased toward sign of beta_2 * corr > 0 here)")
print(f"Full regression  (x1 and x2)   beta_1 ≈ {full[1]:.3f}")

beta_hat = [3.196 2.855]
RSS = 94.54   TSS = 2950.01   R^2 = 0.968   MSE = 2.364

OVB demonstration: true beta_1 = 0.5
Short regression (only x1)      beta_1 ≈ -0.089  (biased toward sign of beta_2 * corr > 0 here)
Full regression  (x1 and x2)   beta_1 ≈ 0.480


## 9. Practice problems

### 9.1 Elasticity
> A regression yields $\log\text{sales} = 4.0 - 1.6\,\log\text{price} + \varepsilon.$ Interpret the slope.

**Answer.** Price elasticity of demand is $-1.6$: a 1% ↑ in price ⇒ 1.6% ↓ in sales. Demand is elastic (|elasticity| > 1).

### 9.2 Categorical interpretation
> `wage ~ education` with `education ∈ {HS, College, Grad}`, reference = `HS`. Output: β_College = 12, β_Grad = 22. Interpret.

**Answer.** A College-educated worker earns \$12 more than an HS-educated worker; a Grad-educated worker earns \$22 more than an HS-educated worker. No direct comparison of College vs. Grad without computing $22-12 = 10$.

### 9.3 Interaction
> `yield ~ fertiliser*irrigation`, output: fertiliser = 0.5, irrigation = 2, fertiliser:irrigation = 1.3. What is the slope on fertiliser (a) under irrigation, (b) without irrigation?

**Answer.** (a) With irrigation ($\mathbb 1=1$): slope = $0.5 + 1.3 = 1.8$. (b) No irrigation: slope = $0.5$. Irrigation roughly quadruples the marginal yield from fertiliser.

### 9.4 OVB sign
> True model: $y=2 x_1 + 3 x_2 + \varepsilon$, and $\text{corr}(x_1, x_2) = -0.4$. If we regress $y$ on $x_1$ only, will $\hat\beta_1$ be above or below 2?

**Answer.** Bias sign = $\text{sign}(\beta_2)\cdot\text{sign}(\text{corr}) = (+)(-) = (-)$. So $\hat\beta_1$ is biased *below* the true value 2.

### 9.5 MC-style — overfitting
> A model has training $R^2=0.95$ but test $R^2=0.30$. What is the main concern? (A) Underfitting (B) Overfitting (C) High bias (D) Data leakage.

**Answer.** **(B) Overfitting** — huge gap between in-sample and out-of-sample fit = classic overfit.

### 9.6 $R^2$ arithmetic
> $n=200$, $\sum(y_i-\bar y)^2 = 800$, RSS = 96. Report MSE and $R^2$.

**Answer.** MSE = $96/200 = 0.48$. $R^2 = 1 - 96/800 = 0.88$.